# RAG Module · Notebook 2 · Embeddings & Vector Search
Turn text into numbers that capture meaning — the building blocks of RAG retrieval.

---
> **Setup:** Run the first two cells before anything else.
> API key is loaded automatically from the `.env` file in the parent folder.

In [1]:
%pip install -q google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, json, math, time, base64, getpass, re, urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

client = genai.Client(api_key=api_key)
MODEL = 'gemma-4-31b-it'
EMBED_MODEL = 'gemini-embedding-001'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen    = client.models.generate_content
_raw_stream = client.models.generate_content_stream
_raw_embed  = client.models.embed_content
_raw_count  = client.models.count_tokens
client.models.generate_content        = lambda *a,**kw: _call_with_retry(_raw_gen,    *a,**kw)
client.models.generate_content_stream = lambda *a,**kw: _call_with_retry(_raw_stream, *a,**kw)
client.models.embed_content           = lambda *a,**kw: _call_with_retry(_raw_embed,  *a,**kw)
client.models.count_tokens            = lambda *a,**kw: _call_with_retry(_raw_count,  *a,**kw)

print('✅ Setup complete. Model:', MODEL, '| Embed model:', EMBED_MODEL, '| Retry-on-429: enabled')

✅ Setup complete. Model: gemma-4-31b-it | Embed model: gemini-embedding-001 | Retry-on-429: enabled


### 🔌 API Key Test

In [3]:
try:
    _r = client.models.generate_content(
        model=MODEL,
        contents="Reply with exactly the words: Hello LLM course",
        config=cfg(temperature=0.0)
    )
    print("✅ API key working!")
    print("Model says:", get_text(_r))
except Exception as e:
    print("❌ API key error:", e)

✅ API key working!
Model says: Hello LLM course


---
## 1. What Are Embeddings? — Meaning as Numbers
An embedding turns any piece of text into a list of numbers that captures its meaning.

**Developer analogy:** A GPS coordinate for meaning — similar texts land near each other in high-dimensional space, the way similar places are close on a map.

```
"Python is great"   → embed() → [0.12, -0.87,  0.34, 0.09, ...] (3072 numbers)
"I love coding"     → embed() → [0.11, -0.82,  0.31, 0.08, ...] (similar!)
"The sky is blue"   → embed() → [-0.43, 0.21, -0.67, 0.55, ...] (very different)
```

> The raw numbers are meaningless on their own — what matters is the **distance** between them.

| Embeddings capture | Embeddings do NOT capture |
|---|---|
| Semantic meaning | Exact wording or phrasing |
| Topic / subject area | Spelling or grammar |
| Sentiment (positive/negative) | Word order within short phrases |
| Context and intent | Formatting or punctuation |

In [4]:
def embed(text: str) -> list:
    """Embed a string using the Gemini embedding model. Returns a list of floats."""
    result = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return result.embeddings[0].values

In [5]:
# Embed three sentences and inspect the raw output
sentences = [
    "Python is a great programming language",
    "I enjoy writing code in Python",
    "The sky is clear and blue today",
]

embeddings = {s: embed(s) for s in sentences}

print(f"Embedding dimension: {len(embeddings[sentences[0]])}\n")
for s, vec in embeddings.items():
    print(f"Text: {s!r}")
    print(f"First 5 values: {[round(v, 4) for v in vec[:5]]}")
    print()

print("⚠️  Raw values look like noise — we need a distance metric to compare them.")

Embedding dimension: 3072

Text: 'Python is a great programming language'
First 5 values: [-0.0034, -0.0057, 0.0098, -0.0525, -0.0116]

Text: 'I enjoy writing code in Python'
First 5 values: [-0.0066, -0.0037, 0.0014, -0.082, -0.0081]

Text: 'The sky is clear and blue today'
First 5 values: [-0.0066, 0.0095, -0.0108, -0.0745, 0.0021]

⚠️  Raw values look like noise — we need a distance metric to compare them.


---
## 2. Cosine Similarity — Measuring How Close Two Meanings Are
Cosine similarity measures the angle between two embedding vectors — 1.0 means identical meaning, 0.0 means unrelated.

**Developer analogy:** Two compass needles pointing in the same direction have zero angle between them — that's similarity = 1.0. Pointing opposite ways = 0.0 (or negative).

```
similarity = (A · B) / (|A| × |B|)

Range:  1.0 = identical meaning
        0.7+ = closely related
        0.5  = somewhat related
        0.3  = loosely related
        0.0  = unrelated
```

In [6]:
def cosine_similarity(a: list, b: list) -> float:
    """Cosine similarity between two embedding vectors. Returns a float in [-1, 1]."""
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x**2 for x in a))
    mag_b = math.sqrt(sum(x**2 for x in b))
    if mag_a == 0 or mag_b == 0:
        return 0.0
    return dot / (mag_a * mag_b)

In [7]:
# Compare 5 sentence pairs: some related, some not
pairs = [
    ("Python is a programming language",   "I write code in Python"),
    ("Python is a programming language",   "JavaScript is used for web development"),
    ("Python is a programming language",   "The weather is warm today"),
    ("Machine learning uses data",         "Neural networks learn from examples"),
    ("Machine learning uses data",         "I enjoy hiking in the mountains"),
]

print(f"{'Sentence A':<42} {'Sentence B':<42} {'Similarity':>10}")
print("-" * 100)
for a, b in pairs:
    vec_a = embed(a)
    vec_b = embed(b)
    score = cosine_similarity(vec_a, vec_b)
    print(f"{a:<42} {b:<42} {score:>10.4f}")

Sentence A                                 Sentence B                                 Similarity
----------------------------------------------------------------------------------------------------


Python is a programming language           I write code in Python                         0.7138


Python is a programming language           JavaScript is used for web development         0.6033


Python is a programming language           The weather is warm today                      0.5433


Machine learning uses data                 Neural networks learn from examples            0.6839


Machine learning uses data                 I enjoy hiking in the mountains                0.5240


In [8]:
# ✏️ Paste any two sentences to see their similarity score
sentence_1 = "Replace this with your first sentence."
sentence_2 = "Replace this with your second sentence."

score = cosine_similarity(embed(sentence_1), embed(sentence_2))
print(f"Similarity: {score:.4f}")
if score > 0.8:
    print("→ Very closely related")
elif score > 0.6:
    print("→ Related")
elif score > 0.4:
    print("→ Somewhat related")
else:
    print("→ Unrelated or different topics")

Similarity: 0.8725
→ Very closely related


---
## 3. Semantic Search — Finding Meaning, Not Just Words
Semantic search finds documents with similar meaning, even when they share no exact words.

**Developer analogy:** Keyword search finds the exact word. Semantic search finds the idea — it works even when the user asks "how do I make my loops faster?" and the answer says "list comprehensions are more performant."

We'll first see the problem with keyword search, then fix it with semantic search.

In [9]:
# A corpus of documents about Python
corpus = [
    "Python was created by Guido van Rossum and first released in 1991.",
    "List comprehensions provide a concise way to create lists and are faster than for-loops.",
    "Python's garbage collector automatically manages memory allocation and deallocation.",
    "The pip package manager installs third-party libraries from PyPI.",
    "Python supports object-oriented programming through classes and inheritance.",
    "Virtual environments isolate project dependencies to avoid version conflicts.",
    "Python's GIL prevents true multi-threading but asyncio enables concurrent I/O.",
    "NumPy arrays are more memory-efficient than Python lists for numerical operations.",
]

query = "how do I speed up my iterations"

# Keyword search — look for exact word overlap
print("=== Keyword search results ===")
query_words = set(query.lower().split())
matches = [(doc, len(query_words & set(doc.lower().split()))) for doc in corpus]
matches.sort(key=lambda x: -x[1])
for doc, count in matches[:3]:
    print(f"  overlap={count}: {doc[:70]}...")

print(f"\n⚠️  Top result has {matches[0][1]} word overlap. Not great.")

=== Keyword search results ===
  overlap=0: Python was created by Guido van Rossum and first released in 1991....
  overlap=0: List comprehensions provide a concise way to create lists and are fast...
  overlap=0: Python's garbage collector automatically manages memory allocation and...

⚠️  Top result has 0 word overlap. Not great.


In [10]:
def semantic_search(query: str, documents: list, top_k: int = 3) -> list:
    """
    Embed the query, compute cosine similarity against all documents,
    return top_k (score, document) pairs sorted by descending score.
    """
    query_vec = embed(query)
    scores = []
    for doc in documents:
        doc_vec = embed(doc)
        score = cosine_similarity(query_vec, doc_vec)
        scores.append((score, doc))
    scores.sort(key=lambda x: -x[0])
    return scores[:top_k]

# Pre-embed the corpus for efficiency
print("Embedding corpus...")
corpus_embeddings = {doc: embed(doc) for doc in corpus}
print(f"Embedded {len(corpus_embeddings)} documents.\n")

# Semantic search
results = semantic_search(query, corpus)

print(f"=== Semantic search: '{query}' ===")
for i, (score, doc) in enumerate(results, 1):
    print(f"\n#{i} (score={score:.4f})")
    print(f"   {doc}")

print("\n✅ Semantic search found the relevant document despite zero word overlap.")

Embedding corpus...


Embedded 8 documents.



=== Semantic search: 'how do I speed up my iterations' ===

#1 (score=0.6238)
   List comprehensions provide a concise way to create lists and are faster than for-loops.

#2 (score=0.5405)
   NumPy arrays are more memory-efficient than Python lists for numerical operations.

#3 (score=0.5323)
   Virtual environments isolate project dependencies to avoid version conflicts.

✅ Semantic search found the relevant document despite zero word overlap.


In [11]:
# ✏️ Add your own document or change the query
my_corpus = corpus + [
    "Add your own document here to see how it ranks.",
]
my_query = "how do I install packages in Python"

results = semantic_search(my_query, my_corpus)
print(f"Query: '{my_query}'\n")
for i, (score, doc) in enumerate(results, 1):
    print(f"#{i} (score={score:.4f}): {doc[:80]}")

Query: 'how do I install packages in Python'

#1 (score=0.7358): The pip package manager installs third-party libraries from PyPI.
#2 (score=0.6234): Virtual environments isolate project dependencies to avoid version conflicts.
#3 (score=0.5818): Python supports object-oriented programming through classes and inheritance.


| | Keyword Search | Semantic Search |
|---|---|---|
| Matches by | Exact word overlap | Meaning and intent |
| Handles synonyms | No | Yes |
| Handles paraphrases | No | Yes |
| Speed | Very fast (grep) | Requires embedding |
| Best for | Known exact terms | Natural language queries |

---
## 4. Bridge to RAG — Where Embeddings Fit In
Embeddings and semantic search are the retrieval engine inside every RAG system.

```
INDEXING (done once when documents are added):
  1. Split documents into chunks          ← covered in Notebook 1
  2. Embed each chunk → vector            ← embed() from this notebook
  3. Store (chunk_text, vector) in DB     ← ChromaDB in Notebook 3

QUERYING (done for every user question):
  1. Embed the user's question → vector   ← embed() from this notebook
  2. Find top-K closest chunk vectors     ← cosine_similarity() from this notebook
  3. Inject retrieved chunks as context   ← grounding from Notebook 1
  4. LLM generates a grounded answer      ← Notebook 3
```

In Notebook 3 you'll replace the in-memory dict with ChromaDB, which handles steps 2–3 automatically at scale.

| Concept | Role in RAG |
|---|---|
| `embed(text)` | Converts text to a searchable vector |
| `cosine_similarity(a, b)` | Measures how relevant a chunk is to the query |
| Semantic search | Finds the right chunks without exact keyword matches |
| ChromaDB | Stores all (chunk, embedding) pairs and handles fast nearest-neighbor lookup |

---
### Key Takeaways

| Concept | One-liner |
|---|---|
| Embedding | Text → dense vector of ~3072 floats encoding meaning |
| Cosine similarity | Angle between vectors — 1.0 = same meaning, 0.0 = unrelated |
| Semantic search | Find by meaning, not exact words |
| EMBED_MODEL | `gemini-embedding-001` — 3072-dimensional vectors |

**Next:** Notebook 3 puts it all together — index a corpus with ChromaDB and build the full RAG pipeline.